# XPert-HG-only Pearson-loss ablation — Kaggle T4
**AIDD8122 Final Project | P2600216 | Qiu Dian**

An exploratory experiment comparing the previously saved HG-only three-term MSE objective (A, trained on Google Colab) with an HG-only variant that adds **0.1 × (1 − Pearson correlation)** across 978 genes (B, trained on Kaggle T4).

This is **not** a complete reproduction of the original UniMol-enabled XPert paper. The data subset, model architecture and other training settings are held fixed as far as possible, but **A and B were trained in different software environments** and only one seed was run per condition. Treat any performance difference as descriptive/exploratory, not a controlled causal estimate.

- Repo: https://github.com/valerieee11/AIDD8122-XPert-HG-Only
- Original XPert: https://github.com/GSanShui/XPert
- Data: https://zenodo.org/records/17182939
- Tested B runtime: **Kaggle Tesla T4, Python 3.13, PyTorch 2.11.0+cu130**.

**Reproduction instruction:** Run from top to bottom in a fresh Kaggle GPU notebook, with **Internet On**. Downloads ~1.6 GB ZIP; requires sufficient space. Runtime package versions and resources might change. The code was assembled from the successful interactive Kaggle cells and syntax checked, but this newly assembled notebook has **not** been end-to-end executed in a fresh GPU runtime.

## 1. Dependencies and fork

In [ ]:
%pip install -q "pandas==2.2.3" "anndata==0.12.7" "bottleneck>=1.5.0" "PyYAML" "h5py" "scipy" "matplotlib"

In [ ]:
from pathlib import Path
import os, sys, subprocess, importlib
import numpy as np
import pandas as pd
import torch
import anndata as ad
import yaml

root = Path('/kaggle/working/XPert')
if not (root / 'train_xpert.py').exists():
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/valerieee11/AIDD8122-XPert-HG-Only.git',
        str(root)
    ], check=True)
os.chdir(root)
(root / 'datasets/__init__.py').touch(exist_ok=True)
if str(root) in sys.path:
    sys.path.remove(str(root))
sys.path.insert(0, str(root))
for name in list(sys.modules):
    if name == 'datasets' or name.startswith('datasets.'):
        sys.modules.pop(name, None)
importlib.invalidate_caches()

print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Enable the GPU accelerator in Kaggle Settings.'
print('GPU:', torch.cuda.get_device_name(0))
for name in [
    'models/model_XPert.py', 'models/model_utils.py', 'utils.py',
    'configs/config_l1000_course_large.yaml',
    'HG_data/saved_embedding/HG_drug_embeddings.npy',
    'processed_data/PPI_gene_vector_128d.npy',
    'results/test_predictions_large.npz'
]:
    assert (root / name).is_file(), f'Missing: {name}'
assert 'scaled_dot_product_attention' in (root / 'models/model_utils.py').read_text(encoding='utf-8')
print('Repository and GPU: ready')

## 2. Restore official data safely
The full processed-data ZIP is large. Extract to a temporary file and atomically replace only when the extraction is finished. If a previous session left a truncated `.h5ad`, this cell repairs it. Verify that 78,453 records × 978 genes can be read.

In [ ]:
import zipfile, shutil
archive = Path('/kaggle/working/processed_data.zip')
full_path = root / 'processed_data/l1000_sdst_78453.h5ad'
full_path.parent.mkdir(parents=True, exist_ok=True)

def verified_h5ad(p):
    if not p.is_file():
        return False
    try:
        d = ad.read_h5ad(p, backed='r')
        good = d.shape == (78453, 978) and 'X_ctl' in d.obsm
        d.file.close()
        return good
    except (OSError, ValueError):
        return False

if not verified_h5ad(full_path):
    if not archive.is_file():
        subprocess.run([
            'wget', '-c', '-O', str(archive),
            'https://zenodo.org/records/17182939/files/processed_data.zip?download=1'
        ], check=True)
    if not zipfile.is_zipfile(archive):
        raise RuntimeError('Downloaded ZIP invalid. Remove and download it again.')
    with zipfile.ZipFile(archive) as zf:
        members = [i for i in zf.infolist()
                   if i.filename.endswith('l1000_sdst_78453.h5ad')]
        if len(members) != 1:
            raise RuntimeError(f'Expected one L1000 SDST file, got {len(members)}')
        info = members[0]
        temp = full_path.with_suffix('.h5ad.partial')
        if temp.exists():
            temp.unlink()
        if shutil.disk_usage(full_path.parent).free < info.file_size + 200*1024**2:
            raise RuntimeError('Not enough free disk space.')
        with zf.open(info) as src, temp.open('wb') as dst:
            shutil.copyfileobj(src, dst, length=8*1024*1024)
        assert temp.stat().st_size == info.file_size, 'Extraction incomplete'
        os.replace(temp, full_path)
assert verified_h5ad(full_path), 'Full L1000 SDST dataset failed verification'
print('Complete L1000 SDST restored:', full_path)

## 3. Recreate identical 1,400-sample subset
The fixed data-sampling seed is 2026. Original `split_1` labels are respected; 1,000 train + 200 validation come from the original train pool and 200 test observations from the original test pool. **This is not a strict drug-disjoint split.** The no-change baseline MSE must be ≈0.519923.

In [ ]:
from scipy import sparse
rng = np.random.default_rng(2026)
full = ad.read_h5ad(full_path, backed='r')
original = full.obs['split_1'].astype(str).to_numpy()
train_pool = np.where(original == 'train')[0]
test_pool = np.where(original == 'test')[0]
assert len(train_pool) >= 1200 and len(test_pool) >= 200

train_valid = rng.choice(train_pool, size=1200, replace=False)
train_idx, valid_idx = train_valid[:1000], train_valid[1000:]
test_idx = rng.choice(test_pool, size=200, replace=False)
selected = np.sort(np.concatenate([train_idx, valid_idx, test_idx]))
mini = full[selected, :].to_memory()
full.file.close()
labels = np.full(len(selected), 'test', dtype=object)
labels[np.isin(selected, train_idx)] = 'train'
labels[np.isin(selected, valid_idx)] = 'valid'
mini.obs['split_course_large'] = labels
assert mini.shape == (1400, 978)
assert mini.obs['split_course_large'].value_counts().to_dict() == {
    'train': 1000, 'valid': 200, 'test': 200
}
mini_path = root / 'processed_data/l1000_sdst_course_1400.h5ad'
mini.write_h5ad(mini_path)
print('Saved dataset:', mini_path)
print('Splits:', mini.obs['split_course_large'].value_counts())
print('Unique drug IDs:', mini.obs['pert_idx'].nunique())
from scipy import sparse
def dense(x):
    return x.toarray() if sparse.issparse(x) else np.asarray(x)
mask = mini.obs['split_course_large'].astype(str).eq('test').to_numpy()
y_test = dense(mini.X[mask]).astype(np.float32)
c_test = dense(mini.obsm['X_ctl'][mask]).astype(np.float32)
base_mse = float(np.mean((y_test - c_test)**2))
print('Baseline test MSE:', base_mse)
assert abs(base_mse - 0.519923210144043) < 1e-4, 'Test selection differs from Experiment A' 

## 4. HG embeddings and UniMol placeholder

In [ ]:
hg = np.load(root/'HG_data/saved_embedding/HG_drug_embeddings.npy', mmap_mode='r')
assert hg.shape == (8981, 256)
assert int(mini.obs['pert_idx'].max()) < hg.shape[0]
placeholder = np.zeros((hg.shape[0], 2, 514), dtype=np.float32)
placeholder[:, 0, 0] = 1.0
np.save(root/'processed_data/course_hg_only_large_placeholder.npy', placeholder)
with (root/'configs/config_l1000_course_large.yaml').open() as f:
    cfg = yaml.safe_load(f)
assert cfg['dataset']['drug_unimol_path'] == 'processed_data/course_hg_only_large_placeholder.npy'
assert cfg['train']['batch_size'] == 1
assert cfg['train']['num_epochs'] == 10
print('HG:', hg.shape, '| placeholder:', placeholder.shape)

## 5. Model loaders and objectives
The original three-component MSE loss is preserved, and **one** extra term is added: 0.1 × (1 − per-sample PCC across genes). Selection of best checkpoint uses **validation MSE**, never test metrics. Batch size is one so Pearson is computed along the gene axis.

In [ ]:
import logging, random
from types import SimpleNamespace
import torch.nn.functional as F
from utils import load_dataloader
from models.model_XPert import XPertNet

device = torch.device('cuda:0')
args = SimpleNamespace(
    model='XPert', dataset='l1000_sdst_course_1400',
    drug_feat='unimol', pretrained_mode='global', mode='train',
    include_cell_idx=True, wo_HG=False, wo_atom=False,
    wo_atom_HG=False, wo_unimol=True, wo_ppi=False,
    use_gene_pos_emed=False, output_attention=False,
    output_cls_embed=False
)
logger = logging.getLogger('xpert_pearson_ablation')
logger.setLevel(logging.INFO)
train_loader, val_loader, test_loader = load_dataloader(
    args, cfg, logger, nfold='split_course_large'
)
assert tuple(len(x.dataset) for x in (train_loader, val_loader, test_loader)) == (1000, 200, 200)
print('Train, valid, test:', *(len(x.dataset) for x in (train_loader, val_loader, test_loader)))

def pearson_correlation_loss(delta_pred, delta_true, eps=1e-6):
    x = delta_pred.float()
    y = delta_true.float()
    x = x - x.mean(dim=-1, keepdim=True)
    y = y - y.mean(dim=-1, keepdim=True)
    numerator = torch.sum(x*y, dim=-1)
    denominator = torch.sqrt(torch.sum(x*x, dim=-1)+eps) * torch.sqrt(torch.sum(y*y, dim=-1)+eps)
    corr = (numerator/denominator).clamp(-1.,1.)
    return 1.0-corr.mean(), corr.mean()

def compute_ablation_loss(outputs):
    trt_direct, ctl_direct, delta_pred, trt_true, ctl_true = outputs[:5]
    delta_true = trt_true - ctl_true
    mse_loss = (F.mse_loss(delta_pred,delta_true)
                + 0.2*F.mse_loss(trt_direct,trt_true)
                + 0.1*F.mse_loss(ctl_direct,ctl_true))
    pearson_loss, corr = pearson_correlation_loss(delta_pred,delta_true)
    return mse_loss+0.1*pearson_loss, mse_loss, pearson_loss, corr

@torch.no_grad()
def evaluate_validation(model, loader):
    model.eval()
    squared, count, correlations = 0.0, 0, []
    for batch in loader:
        o = model(batch)
        yt, yc, dp = o[3], o[4], o[2]
        squared += ((yt-(yc+dp))**2).sum().item()
        count += yt.numel()
        dt = yt-yc
        for a,b in zip(dt,dp):
            a,b=a.float().flatten(),b.float().flatten()
            if a.std().item()>1e-8 and b.std().item()>1e-8:
                a,b=a-a.mean(),b-b.mean()
                correlations.append(((a*b).sum()/(torch.linalg.vector_norm(a)*torch.linalg.vector_norm(b))).item())
    return squared/count, float(np.mean(correlations)) if correlations else np.nan
print('Pearson-loss ablation: ready')

## 6. Train Experiment B and save a ZIP immediately
**Warning:** This cell takes GPU resources and trains 10 epochs. The ZIP is saved in temporary `/kaggle/working`; **download it before ending the session**, or save an output-bearing Kaggle notebook version. No pretrained original A checkpoint is required for B.

In [ ]:
import json
from zipfile import ZipFile, ZIP_DEFLATED
out_dir = root/'experiment/pearson_ablation'
out_dir.mkdir(parents=True,exist_ok=True)
random.seed(2026)
np.random.seed(2026)
torch.manual_seed(2026)
torch.cuda.manual_seed_all(2026)

model_b = XPertNet(args,cfg,device,logger)
model_b.init_weights()
model_b=model_b.to(device)
optim_b=torch.optim.Adam(model_b.parameters(),lr=1e-4,weight_decay=1e-5)
model_file=out_dir/'xpert_hg_pearson_best.pt'
history_file=out_dir/'training_history_pearson.csv'
best_val=float('inf')
best_epoch=None
history=[]
for epoch in range(1,11):
    model_b.train()
    totals,mses,pearsons=[],[],[]
    for step,batch in enumerate(train_loader,1):
        optim_b.zero_grad(set_to_none=True)
        total,mse,ploss,pcc=compute_ablation_loss(model_b(batch))
        if not torch.isfinite(total):
            raise RuntimeError(f'Non-finite loss at epoch {epoch}, step {step}')
        total.backward()
        torch.nn.utils.clip_grad_norm_(model_b.parameters(),max_norm=1.0)
        optim_b.step()
        totals.append(total.item())
        mses.append(mse.item())
        pearsons.append(ploss.item())
        if step%250==0:
            print(f'Epoch {epoch}/10: batch {step}/{len(train_loader)}',flush=True)
    val_mse,val_corr=evaluate_validation(model_b,val_loader)
    history.append({'epoch':epoch,'train_total_loss':float(np.mean(totals)),
                    'train_mse_loss':float(np.mean(mses)),
                    'train_pearson_loss':float(np.mean(pearsons)),
                    'val_mse':val_mse,'val_pearson_delta':val_corr})
    pd.DataFrame(history).to_csv(history_file,index=False)
    if val_mse<best_val:
        best_val,best_epoch=val_mse,epoch
        torch.save(model_b.state_dict(),model_file)
    print(f'Epoch {epoch:02d}: validation MSE={val_mse:.6f}, PCC={val_corr:.6f}',flush=True)
summary={'experiment':'XPert-HG-only + Pearson loss', 'seed':2026,
         'pearson_loss_weight':0.1,'epochs':10,'best_epoch':best_epoch,
         'best_validation_mse':best_val,'pytorch_version':torch.__version__,
         'device':torch.cuda.get_device_name(0)}
summary_file=out_dir/'training_summary.json'
summary_file.write_text(json.dumps(summary,indent=2),encoding='utf-8')
training_backup=Path('/kaggle/working/XPert_Pearson_Backup.zip')
with ZipFile(training_backup,'w',ZIP_DEFLATED) as zf:
    for f in [model_file,history_file,summary_file]:
        zf.write(f,arcname=f.name)
print('Training complete; best epoch:',best_epoch,'best validation MSE:',best_val)
print('Backup:',training_backup,'size MB:',round(training_backup.stat().st_size/1024**2,2))

## 7. Evaluate held-out test observations against saved Experiment A
Experiment A predictions are loaded from the repository. **Exact per-observation alignment** is checked by comparing both the true treated expression and the control expression, before comparing A/B. A failed alignment assertion means stop; do not remove it.

In [ ]:
state=torch.load(model_file,map_location=device,weights_only=True)
model_b.load_state_dict(state)
model_b.eval()
yt_list,yp_list,yc_list=[],[],[]
with torch.no_grad():
    for batch in test_loader:
        o=model_b(batch)
        yt_list.append(o[3].cpu().numpy())
        yc_list.append(o[4].cpu().numpy())
        yp_list.append((o[4]+o[2]).cpu().numpy())
y_true_b=np.concatenate(yt_list)
y_ctl_b=np.concatenate(yc_list)
y_pred_b=np.concatenate(yp_list)
assert y_true_b.shape == y_ctl_b.shape == y_pred_b.shape == (200,978)
assert np.isfinite(y_pred_b).all()
with np.load(root/'results/test_predictions_large.npz') as old_result:
    y_true_a=old_result['y_true'].copy()
    y_pred_a=old_result['y_pred'].copy()
    y_ctl_a=old_result['y_control'].copy()
    drug_ids=old_result['drug_ids'].copy()
assert np.allclose(y_true_a,y_true_b,atol=1e-5,rtol=1e-5), 'Different test samples/order'
assert np.allclose(y_ctl_a,y_ctl_b,atol=1e-5,rtol=1e-5), 'Control expression differs'
assert abs(np.mean((y_true_b-y_ctl_b)**2)-0.51992321)<1e-4

def mean_pearson_delta(y_true,y_pred,y_ctl):
    cors=[]
    for yt,yp,yc in zip(y_true,y_pred,y_ctl):
        dt,dp=yt-yc,yp-yc
        if np.std(dt)>=1e-8 and np.std(dp)>=1e-8:
            cors.append(np.corrcoef(dt,dp)[0,1])
    return float(np.mean(cors)) if cors else np.nan

def calculate_metrics(name,pred):
    mse=float(np.mean((y_true_b-pred)**2))
    return {'Model':name,'MSE':mse,'RMSE':float(np.sqrt(mse)),
            'MAE':float(np.mean(np.abs(y_true_b-pred))),
            'Pearson_Delta':mean_pearson_delta(y_true_b,pred,y_ctl_b),
            'Mean_Abs_Change':float(np.mean(np.abs(pred-y_ctl_b)))}
comparison=pd.DataFrame([
    calculate_metrics('No-change Baseline',y_ctl_b),
    calculate_metrics('Experiment A: MSE-only',y_pred_a),
    calculate_metrics('Experiment B: MSE + Pearson',y_pred_b)])
print(comparison.to_string(index=False))
comparison.to_csv(out_dir/'ablation_comparison.csv',index=False)
np.savez_compressed(out_dir/'test_predictions_pearson.npz',
                    y_true=y_true_b,y_pred=y_pred_b,y_control=y_ctl_b,drug_ids=drug_ids)
print('A vs B MSE improvement:',float(comparison.iloc[1]['MSE']-comparison.iloc[2]['MSE']))

## 8. Paired bootstrap clustered by drug ID
Resample drug IDs (196 unique identifiers in the recorded experiment) rather than treating all 200 records as independent. This captures test-set variation from repeated drugs, **not training variation or cell/batch dependence**. Store all replicates and summaries.

In [ ]:
mse_a=((y_true_b-y_pred_a)**2).mean(axis=1)
mse_b=((y_true_b-y_pred_b)**2).mean(axis=1)
improvement=mse_a-mse_b
ids=np.asarray(drug_ids).reshape(-1).astype(str)
unique_drugs,group=np.unique(ids,return_inverse=True)
group_sum=np.bincount(group,weights=improvement)
group_count=np.bincount(group)
rng=np.random.default_rng(2026)
boot=np.empty(5000)
for i in range(5000):
    sampled=rng.integers(0,len(unique_drugs),size=len(unique_drugs))
    boot[i]=group_sum[sampled].sum()/group_count[sampled].sum()
low,high=np.percentile(boot,[2.5,97.5])
print('Unique drugs:',len(unique_drugs))
print('Observed MSE reduction:',improvement.mean())
print('Percentile 95% CI:',low,high)
print('Fraction positive:',np.mean(boot>0))
pd.DataFrame({'bootstrap_mse_reduction':boot}).to_csv(out_dir/'ablation_bootstrap.csv',index=False)
pd.DataFrame([{'n_test_samples':200,'n_unique_drugs':len(unique_drugs),
               'bootstrap_repetitions':5000,
               'observed_mse_reduction':float(improvement.mean()),
               'ci_lower_95':float(low),'ci_upper_95':float(high),
               'fraction_positive_resamples':float(np.mean(boot>0))}]).to_csv(
                   out_dir/'ablation_bootstrap_summary.csv',index=False)

## 9. Final durable backup
Archive checkpoint, metrics, test predictions and bootstrap replicates. **Download the ZIP to your computer** using Kaggle's output/file-browser menu, and check it opens before closing the session. GitHub intentionally stores the lightweight metrics/plots and not the bulky model checkpoint.

In [ ]:
final_zip=Path('/kaggle/working/XPert_Ablation_Complete.zip')
file_names=['xpert_hg_pearson_best.pt','training_history_pearson.csv',
            'training_summary.json','ablation_comparison.csv',
            'test_predictions_pearson.npz','ablation_bootstrap.csv',
            'ablation_bootstrap_summary.csv']
with ZipFile(final_zip,'w',ZIP_DEFLATED) as zf:
    for name in file_names:
        p=out_dir/name
        assert p.is_file(),f'Missing: {p}'
        zf.write(p,arcname=name)
with ZipFile(final_zip) as zf:
    assert zf.testzip() is None
print('FINAL BACKUP READY', final_zip, 'size MB:',round(final_zip.stat().st_size/1024**2,2))